In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_module import PhoDataset, Vocabulary, collate_fn
from src.models.model01 import Encoder, Decoder, Model01

In [23]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import pandas as pd
from tqdm import tqdm

from torchmetrics.text.rouge import ROUGEScore

# 1. Preparation

## 1.1. Vocabulary

In [3]:
vocab = Vocabulary(
    r'..\data\small-train.json',
    freq_threshold=1
)

                                                 english  \
0                           It begins with a countdown .   
1      On August 14th , 1947 , a woman in Bombay goes...   
2      Across India , people hold their breath for th...   
3      And at the stroke of midnight , a squirming in...   
4      These events form the foundation of " Midnight...   
...                                                  ...   
19995               And the man was incredibly curious .   
19996  And he wanted to understand what it was and wh...   
19997                    And one day , we were walking .   
19998               We were in France , in Les Houches .   
19999               We were up in the mountains , 1976 .   

                                              vietnamese  
0             Câu chuyện bắt đầu với buổi lễ đếm ngược .  
1      Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở...  
2      Cùng lúc , trên khắp đất Ấn , người ta nín thở...  
3      Khi đồng hồ điểm thời khắc nửa đêm ,

## 1.2. Model

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


### 1.2.1. `Encoder` & `Decoder`

In [5]:
# --- Encoder ---
encoder = Encoder(
    vocab.num_en_words, 
    256, 
    256, 
    3, 
    0.2
).to(device)
print(encoder)

# --- Decoder ---
decoder = Decoder(
    vocab.num_vi_words, 
    256, 
    256, 
    vocab.num_vi_words, 
    3, 
    0.2
).to(device)
print(decoder)

Encoder(
  (embedding): Embedding(17418, 256)
  (rnn): LSTM(256, 256, num_layers=3, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
)
Decoder(
  (embedding): Embedding(7061, 256)
  (rnn): LSTM(256, 256, num_layers=3, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=7061, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


### 1.2.2. `LSTM` model with `Encoder` and `Decoder`

In [6]:
model = Model01(
    encoder,
    decoder,
    device
).to(device)
print(model)

Model01(
  (encoder): Encoder(
    (embedding): Embedding(17418, 256)
    (rnn): LSTM(256, 256, num_layers=3, batch_first=True, dropout=0.2)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(7061, 256)
    (rnn): LSTM(256, 256, num_layers=3, batch_first=True, dropout=0.2)
    (fc): Linear(in_features=256, out_features=7061, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
)


## 1.3. Dataset

In [7]:
def my_collate_fn(batch):
    return collate_fn(batch, padding_value=vocab.en2i['<PAD>'])

In [8]:
train_df = PhoDataset(
    '..\data\small-train.json',
    vocab
)
train_loader = DataLoader(
    train_df,
    batch_size = 64,
    shuffle=True,
    collate_fn=my_collate_fn,
    num_workers=0,
)

In [9]:
dev_df = PhoDataset(
    '..\data\small-dev.json',
    vocab
)
dev_loader = DataLoader(
    dev_df,
    batch_size = 64,
    shuffle=False,
    collate_fn=my_collate_fn,
    num_workers=0,
)

In [10]:
test_df = PhoDataset(
    '..\data\small-test.json',
    vocab
)
test_loader = DataLoader(
    test_df,
    batch_size = 64,
    shuffle=False,
    collate_fn=my_collate_fn,
    num_workers=0,
)

# 2. Training

## 2.1. Epoch

### 2.1.1. Training epoch

In [11]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab.vi2i['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
def train_epoch(model, loader, optimizer, criterion, device, clip=1.0):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(loader, desc='Training', leave=True)
    
    for batch in progress_bar:
        src = batch['encoder_input'].to(device)
        trg = batch['decoder_input'].to(device)
        
        optimizer.zero_grad()

        output = model(src, trg[:, :-1])
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        
        trg_target = trg[:, 1:].reshape(-1)
        loss = criterion(output, trg_target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    return epoch_loss / len(loader)

### 2.1.2. Evaluating epoch

In [13]:
def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        progress_bar = tqdm(loader, desc='Evaluating', leave=True)
        
        for batch in progress_bar:
            src = batch['encoder_input'].to(device)
            trg = batch['decoder_input'].to(device)

            output = model(src, trg[:, :-1], teacher_forcing_ratio=0)
            
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            trg_target = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg_target)
            epoch_loss += loss.item()
            
            progress_bar.set_postfix({'val_loss': loss.item()})
            
    return epoch_loss / len(loader)

## 2.2 Training model

In [14]:
N_EPOCHS = 10


In [15]:
def training(
    model,
    train_loader,
    dev_loader,
    optimizer,
    criterion,
    device,
    num_epochs = 10,
    best_valid_loss = float('inf')
):
    training_loss = []
    evaluating_loss = []

    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        valid_loss = evaluate(model, dev_loader, criterion, device)
        
        training_loss.append(train_loss)
        evaluating_loss.append(valid_loss)
        
        print(f'\n' + '='*30)
        print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.4f} | Val. Loss: {valid_loss:.4f}')
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            import os
            os.makedirs('../checkpoints/', exist_ok=True)
            
            torch.save(model.state_dict(), '../checkpoints/best_model01.pt')
            print(f"\t [!] New Best Valid Loss. Model saved.")
        print('='*30 + '\n')

        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return training_loss, evaluating_loss

In [33]:
result = training(model, train_loader, dev_loader, optimizer, criterion, device)

Evaluating: 100%|██████████| 32/32 [00:07<00:00,  4.54it/s, val_loss=6.03]



Epoch: 01 | Train Loss: 5.9846 | Val. Loss: 6.0881
	 [!] New Best Valid Loss. Model saved.



Evaluating: 100%|██████████| 32/32 [00:03<00:00,  9.22it/s, val_loss=6.04]



Epoch: 02 | Train Loss: 5.9107 | Val. Loss: 6.0729
	 [!] New Best Valid Loss. Model saved.



Evaluating: 100%|██████████| 32/32 [00:02<00:00, 12.51it/s, val_loss=6.02]


...


Epoch: 06 | Train Loss: 5.6901 | Val. Loss: 5.9815
	 [!] New Best Valid Loss. Model saved.



Evaluating: 100%|██████████| 32/32 [00:06<00:00,  4.95it/s, val_loss=5.99]



Epoch: 07 | Train Loss: 5.6399 | Val. Loss: 5.9489
	 [!] New Best Valid Loss. Model saved.



Training:  72%|███████▏  | 225/313 [03:00<01:10,  1.25it/s, loss=5.68]


RuntimeError: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR

# 3. Evaluating and using model

## 3.1. Evaluating

In [ ]:
def evaluate_rouge(model, loader, vocab, device, max_len=50):
    model.eval()
    rouge_metric = ROUGEScore(rouge_keys='rougeL')
    
    predictions = []
    references = []
    progress_bar = tqdm(loader, desc='Calculating ROUGE-L', leave=True)

    with torch.no_grad():
        for batch in progress_bar:
            src = batch['encoder_input'].to(device)
            trg = batch['decoder_input']
            
            for i in range(src.shape[0]):
                single_src = src[i:i+1]
                
                hidden, cell = model.encoder(single_src)
                
                trg_indexes = [vocab.vi2i['<SOS>']]
                
                for _ in range(max_len):
                    trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
                    output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
                    pred_token = output.argmax(1).item()
                    trg_indexes.append(pred_token)
                    if pred_token == vocab.vi2i['<EOS>']:
                        break
                
                def tokens_to_sentence(ids, vocabulary, is_vi=True):
                    specials = {vocabulary.vi2i['<SOS>'], vocabulary.vi2i['<EOS>'], vocabulary.vi2i['<PAD>']}
                    if is_vi:
                        return " ".join([vocabulary.i2vi[idx] for idx in ids if idx not in specials])
                    else:
                        return " ".join([vocabulary.i2en[idx] for idx in ids if idx not in specials])

                pred_sentence = tokens_to_sentence(trg_indexes, vocab)
                actual_sentence = tokens_to_sentence(trg[i].tolist(), vocab)
                
                predictions.append(pred_sentence)
                references.append(actual_sentence)

    results = rouge_metric(predictions, references)
    return results['rougeL_fmeasure'].item()

In [19]:
model.load_state_dict(torch.load('../checkpoints/best_model01.pt'))

print("--- Bắt đầu tính điểm ROUGE-L ---")
score = evaluate_rouge(model, test_loader, vocab, device)

print(f"\n" + "="*40)
print(f"KẾT QUẢ ĐÁNH GIÁ (TEST SET)")
print(f"ROUGE-L Score: {score:.4f} ({score*100:.2f}%)")
print("="*40)

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_24992\4294230467.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../checkpoints/best_model

--- Bắt đầu tính điểm ROUGE-L ---


Calculating ROUGE-L:   0%|          | 0/32 [00:00<?, ?it/s]

Calculating ROUGE-L: 100%|██████████| 32/32 [00:56<00:00,  1.76s/it]



KẾT QUẢ ĐÁNH GIÁ (TEST SET)
ROUGE-L Score: 0.2236 (22.36%)


In [21]:
def translate_sentence(sentence, model=model, vocab=vocab, device=device, max_len=50):
    """
    Hàm dịch một câu tiếng Anh sang tiếng Việt.
    """
    model.eval()

    if isinstance(sentence, str):
        tokens = [vocab.en2i.get(token.lower(), vocab.en2i['<UNK>']) for token in sentence.split()]
    else:
        tokens = sentence 

    tokens = [vocab.en2i['<SOS>']] + tokens + [vocab.en2i['<EOS>']]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
    
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    trg_indexes = [vocab.vi2i['<SOS>']]
    
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == vocab.vi2i['<EOS>']:
            break
    
    translated_tokens = [vocab.i2vi.get(i, '<UNK>') for i in trg_indexes]
    
    return " ".join(translated_tokens[1:-1])

In [41]:
raw_sentence = 'He is a student .'
translated_sentence = translate_sentence(raw_sentence)

print(f'English      {raw_sentence}')
print(f'Vietnamese   {translated_sentence}')

English      He is a student .
Vietnamese   là là là một . . . .
